In [1]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from itertools import product

import warnings
warnings.filterwarnings("ignore")

def warn(*args, **kwargs):
    return None

warnings.warn = warn

In [2]:
env = gym.make('CliffWalking-v0')

In [3]:
num_actions = env.action_space.n
num_states = env.observation_space.n

print(num_actions)
print(num_states)

4
48


In [4]:
env.reset()
action = env.action_space.sample()
print(f"Action {action}")
obs, reward, done, _ = env.step(action)
print(f"State: {obs}\nReward: {reward}\nTerminate: {done}")

Action 3
State: 36
Reward: -1
Terminate: False


In [5]:
def run_episodes(env, agent, total_episodes, max_steps, state_visits):
    """
    Executa múltiplos episódios para treinar e avaliar o agente.
    Também rastreia as visitas aos estados.

    Parâmetros:
    env: O ambiente do Gym.
    agent: O agente que está sendo testado.
    total_episodes: Número total de episódios a serem executados.
    max_steps: Número máximo de passos por episódio.
    state_visits: Array para contar as visitas a cada estado.

    Retorna:
    rewards: Lista de recompensas acumuladas por episódio.
    """
    rewards = []
    for _ in tqdm(range(total_episodes)):
        obs = env.reset()
        state_visits[obs] += 1
        total_reward = 0
        for _ in range(max_steps):
            action = agent.agent_start(obs) if total_reward == 0 else agent.agent_step(reward, obs)
            obs, reward, terminated, _ = env.step(action)
            state_visits[obs] += 1
            total_reward += reward
            if terminated:
                agent.agent_end(reward)
                break
        rewards.append(total_reward)
    print(f"Recompensa Acumulada: {sum(rewards)}")
    return rewards

In [6]:
def plot_rewards_cumulative(rewards):
    cumulative_rewards = [sum(rewards[:i+1]) for i in range(len(rewards))]

    plt.figure(figsize=(12, 8))
    plt.plot(cumulative_rewards)
    plt.xlabel("Episódios")
    plt.ylabel("Recompensa Acumulada")
    plt.title("Recompensa Acumulada ao Longo dos Episódios")
    plt.grid(True)
    plt.show()

In [7]:
def plot_state_visitations(state_visits, grid_shape):
    """
    Plota as visitas aos estados em um ambiente Cliff Walking.

    Parâmetros:
    state_visits: Array de contagem das visitas a cada estado.
    grid_shape: Tupla indicando a forma da grade do ambiente (filas, colunas).
    """
    state_visits_grid = state_visits.reshape(grid_shape)

    plt.figure(figsize=(12, 8))
    plt.imshow(state_visits_grid, cmap='hot', interpolation='nearest')
    plt.colorbar(label='Número de Visitas')
    plt.xlabel('Colunas')
    plt.ylabel('Filas')
    plt.title('Visitas aos Estados no Ambiente Cliff Walking')
    plt.show()

In [8]:
class ExpectedSarsaAgent:
    def __init__(self, env, num_actions, num_states, discount, epsilon, step_size, seed=42):
      # Store the parameters provided in agent_init_info.
      self.num_actions = num_actions
      self.num_states = num_states
      self.epsilon = epsilon
      self.step_size = step_size
      self.discount = discount
      self.seed = seed
      self.rand_generator = np.random.RandomState(seed)
      # Create an array for action-value estimates and initialize it to zero.
      self.q = np.zeros((self.num_states, self.num_actions)) # Matriz de Q-value

    def agent_start(self, observation):
        """The first method called when the episode starts, called after
        the environment starts.
        Args:
            observation (int): the state observation from the
                environment's evn_start function.
        Returns:
            action (int): the first action the agent takes.
        """

        # Choose action using epsilon greedy.
        state = observation
        current_q = self.q[state, :]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)
        self.prev_state = state
        self.prev_action = action
        return action

    def agent_step(self, reward, observation):
        """A step taken by the agent.
        Args:
            reward (float): the reward received for taking the last action taken
            observation (int): the state observation from the
                environment's step based on where the agent ended up after the
                last step.
        Returns:
            action (int): the action the agent is taking.
        """

        # Choose action using epsilon greedy.
        state = observation
        current_q = self.q[state,:]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)

        expected_q = np.sum((self.epsilon / self.num_actions) * current_q)
        best_action = self.argmax(current_q)

        expected_q += (1.0 - self.epsilon) * self.q[state, best_action]

        self.q[self.prev_state, self.prev_action] += self.step_size * (
            reward + self.discount * expected_q - self.q[self.prev_state, self.prev_action])

        self.prev_state = state
        self.prev_action = action
        return action

    def agent_end(self, reward):
        """Run when the agent terminates.
        Args:
            reward (float): the reward the agent received for entering the
                terminal state.
        """
        # Update the Q-value for the last state-action pair.
        self.q[self.prev_state, self.prev_action] += self.step_size * (reward - self.q[self.prev_state, self.prev_action])

    def argmax(self, q_values):
        """argmax with random tie-breaking
        Args:
            q_values (Numpy array): the array of action-values
        Returns:
            action (int): an action with the highest value
        """
        top = float("-inf")
        ties = []

        for i in range(len(q_values)):
            if q_values[i] > top:
                top = q_values[i]
                ties = []

            if q_values[i] == top:
                ties.append(i)

        return self.rand_generator.choice(ties)

# Recompensa
Cada etapa gera -1 de recompensa, a menos que o jogador pise no penhasco, o que gera -100 de recompensa.

In [ ]:
# Inicialização do agente
num_actions = env.action_space.n
num_states = env.observation_space.n
discount = 0.999
epsilon = 0.01
step_size = 0.99

agent_info = (env, num_actions, num_states, discount, epsilon, step_size)
agent = ExpectedSarsaAgent(*agent_info)
total_episodes = 1000
max_steps = 100
state_visits = np.zeros(num_states)  # Array para rastrear visitas aos estados
rewards = run_episodes(env, agent, total_episodes, max_steps, state_visits)
env.close()
plot_rewards_cumulative(rewards)

In [ ]:
env_shape = (4, 12)
plot_state_visitations(state_visits, env_shape)